# 1. 멀티에이전트 상태 소유권

**시나리오:** 운영 장애를 triage·investigator·commander·risk guard 역할이 순서대로 검토합니다.

**학습 목표:** `IncidentState`에서 각 Agent가 어떤 필드를 만들고 다음 역할에 넘기는지 구분합니다.

## 중요 변수·함수

- `triage`: objective와 변경 연관성 판단을 소유합니다.
- `findings`: investigator가 runbook 근거로 작성합니다.
- `proposals`: commander가 만들지만 실행 권한은 없습니다.
- `agents_run`: 역할 실행 순서를 남기는 audit trace입니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# canonical incident graph를 실행합니다.
from week3.app import IncidentRequest, create_fixture_services, run_incident_response

incident = IncidentRequest(service='checkout', summary='Errors after deployment', severity='SEV1')
result = run_incident_response(incident, create_fixture_services())
result['agents_run']

In [ ]:
# 역할 경계와 비실행 불변조건을 확인합니다.
assert result['agents_run'][0:3] == ['triage_agent', 'investigator_agent', 'commander_agent']
assert result['executed_commands'] == []
{'status': result['status'], 'citations': result['citations']}

## 예측 과제와 해석

**예측 과제:** commander가 직접 `executed_commands`를 채우지 못하게 해야 하는 이유를 적으세요.

**해석:** 멀티에이전트는 역할 이름을 늘리는 것이 아니라 state 쓰기 권한과 책임을 분리하는 설계입니다.